# V1-S14 — Forecasting evaluation on the SEALED test set (Gate G4)

This notebook **loads** the V1-S14 sealed-test-set evaluation artifacts produced **once** by
`scifield forecasting gnn-eval` and renders the **test** metrics table, the HGT−`no_graph`
ablation, the pre-registered Wilcoxon (H2), the calibration/reliability diagram, the publication
figure **F3**, and the descriptive label-sensitivity table.

It performs **no network calls**, **does NOT recompute the Gate G4 verdict** (that is frozen in
`data/v1/forecasting_test_wilcoxon.json`), and **does NOT touch the sealed test set, build
snapshots, or load/score any model** — all of that already happened in the single controlled
`gnn-eval` run. It only reads `forecasting_test_{metrics,per_unit,calibration,sensitivity}.parquet`,
`forecasting_test_wilcoxon.json`, and their `.run.json` sidecars.

**The honest outcome is a NULL finding.** On the sealed test (forecast-ORIGIN years 2021–2022,
n=138, 13 positives) the HGT scores emergence AUC **0.781 vs. `no_graph` 0.804** — a **−2.34 pp**
graph contribution (H1 **FAIL**) — and the paired per-row Brier-loss Wilcoxon is significant **in
favour of the baseline** (p≈2.6e-11; HGT significantly worse-calibrated). `overall_pass = False`.

**Downstream = the human Gate G4 decision.** Samer adjudicates and signs
`docs/gates/G4_forecasting.md`; V1-S15 is gated on that decision. A clean null is publishable
(plan §6) — F3 stands as "graph structure did not improve 3-yr emergence forecasting over a
graph-free baseline on the sealed test."

## 1. Setup

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

DATA = repo_root / "data" / "v1"
FIGURES_DIR = repo_root / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DPI = 120

from scifield.repro import record_run  # noqa: E402


def _load_parquet(path: Path) -> pd.DataFrame | None:
    """Defensive parquet read — return None (with a message) if absent."""
    if not path.exists():
        print(f"MISSING artifact: {path} — cannot render this section.")
        return None
    return pd.read_parquet(path)


def _load_json(path: Path) -> dict | None:
    """Defensive sidecar read — return None (with a message) if absent."""
    if not path.exists():
        print(f"MISSING sidecar: {path} — provenance fields unavailable.")
        return None
    return json.loads(path.read_text())


metrics = _load_parquet(DATA / "forecasting_test_metrics.parquet")
per_unit = _load_parquet(DATA / "forecasting_test_per_unit.parquet")
calib = _load_parquet(DATA / "forecasting_test_calibration.parquet")
sens = _load_parquet(DATA / "forecasting_test_sensitivity.parquet")
wilcoxon = _load_json(DATA / "forecasting_test_wilcoxon.json")
metrics_sidecar = _load_json(DATA / "forecasting_test_metrics.parquet.run.json")
print(
    "loaded:",
    {
        k: ("ok" if v is not None else None)
        for k, v in {
            "metrics": metrics,
            "per_unit": per_unit,
            "calib": calib,
            "sens": sens,
            "wilcoxon": wilcoxon,
            "metrics_sidecar": metrics_sidecar,
        }.items()
    },
)

loaded: {'metrics': 'ok', 'per_unit': 'ok', 'calib': 'ok', 'sens': 'ok', 'wilcoxon': 'ok', 'metrics_sidecar': 'ok'}


## 2. Test metrics table (emergence AUC + share MAPE, 5 models)

In [2]:
N_TEST = int(metrics["n_test"].iloc[0])
N_TEST_POS = int(metrics["n_test_pos"].iloc[0])
N_TRAIN = int(metrics["n_train"].iloc[0])
print(
    f"Sealed test set: forecast-ORIGIN years 2021-2022 | n_train={N_TRAIN} "
    f"n_test={N_TEST} n_test_pos={N_TEST_POS} (pos rate {N_TEST_POS / N_TEST:.1%})"
)
print(f"Small-sample caveat: with only {N_TEST_POS} positives, per-model AUCs are high-variance.")
metrics.round(4)

Sealed test set: forecast-ORIGIN years 2021-2022 | n_train=1298 n_test=138 n_test_pos=13 (pos rate 9.4%)
Small-sample caveat: with only 13 positives, per-model AUCs are high-variance.


,model,emergence_auc,share_mape,n_train,n_test,n_test_pos,fallback_frac
0,naive,0.4074,0.2605,1298,138,13,NaN
1,arima,0.6529,0.2744,1298,138,13,0.1449
2,mlp,0.7009,0.3954,1298,138,13,NaN
3,no_graph,0.8043,0.4610,1298,138,13,NaN
4,hgt,0.7809,0.5729,1298,138,13,NaN


## 3. Ablation — HGT vs no_graph (graph-structure contribution, PR2 §6)

In [3]:
auc = metrics.set_index("model")["emergence_auc"]
auc_hgt = float(auc["hgt"])
auc_ng = float(auc["no_graph"])
delta_pp = (auc_hgt - auc_ng) * 100
print(f"HGT emergence AUC      = {auc_hgt:.4f}")
print(f"no_graph emergence AUC = {auc_ng:.4f}  (best baseline on test; PR2 ablation)")
print(f"Ablation delta (HGT - no_graph) = {delta_pp:+.2f} pp")
print("Pre-registered H1 bar: HGT must exceed no_graph by > +5.0 pp.")
print("Result: the graph structure did NOT help — HGT is slightly WORSE than")
print("        the graph-free baseline.")

HGT emergence AUC      = 0.7809
no_graph emergence AUC = 0.8043  (best baseline on test; PR2 ablation)
Ablation delta (HGT - no_graph) = -2.34 pp
Pre-registered H1 bar: HGT must exceed no_graph by > +5.0 pp.
Result: the graph structure did NOT help — HGT is slightly WORSE than
        the graph-free baseline.


## 4. Wilcoxon (H2) — paired per-row Brier-loss, HGT vs no_graph

In [4]:
pb = wilcoxon["primary_brier"]
rs = wilcoxon["secondary_raw_score"]
vd = wilcoxon["verdict"]

print("PRIMARY (pre-registered intent): per-row Brier-loss paired Wilcoxon, lower = better")
print(f"  statistic={pb['statistic']:.1f}  p={pb['pvalue']:.3e}  n_pairs={pb['n_pairs']}")
print(
    f"  median(brier_hgt - brier_no_graph) = {pb['median_diff']:+.4f}  "
    f"direction = {pb['direction'].upper()}"
)
print("  -> p < 0.05 is TRUE, but the significance FAVOURS THE BASELINE:")
print("     HGT is significantly WORSE-calibrated (higher Brier). This does NOT support F3.")
print()
print("SECONDARY (robustness): raw emergence-score paired Wilcoxon")
print(
    f"  statistic={rs['statistic']:.1f}  p={rs['pvalue']:.3e}  "
    f"median_diff={rs['median_diff']:+.4f}  direction = {rs['direction']}"
)
print()
print("VERDICT (frozen, mechanical — computed by gate_g4_verdict, NOT recomputed here):")
print(f"  H1 (delta > +5pp AUC): {vd['h1_pass']}   (delta = {vd['delta_pp']:+.2f} pp)")
print(f"  H2 (Brier p < 0.05):   {vd['h2_pass']}   direction = {vd['h2_direction']}")
print(f"  overall_pass = {vd['overall_pass']}   ->   {vd['mechanical_recommendation']}")

PRIMARY (pre-registered intent): per-row Brier-loss paired Wilcoxon, lower = better
  statistic=1659.0  p=2.629e-11  n_pairs=138
  median(brier_hgt - brier_no_graph) = +0.2330  direction = FAVORS_BASELINE
  -> p < 0.05 is TRUE, but the significance FAVOURS THE BASELINE:
     HGT is significantly WORSE-calibrated (higher Brier). This does NOT support F3.

SECONDARY (robustness): raw emergence-score paired Wilcoxon
  statistic=0.0  p=2.156e-24  median_diff=+0.4969  direction = model_higher

VERDICT (frozen, mechanical — computed by gate_g4_verdict, NOT recomputed here):
  H1 (delta > +5pp AUC): False   (delta = -2.34 pp)
  H2 (Brier p < 0.05):   True   direction = favors_baseline
  overall_pass = False   ->   NULL FINDING (F3 reported as null)


## 5. Calibration / reliability

In [5]:
print("Reliability bins (non-empty), equal-width over [0, 1]:")
shown = calib[calib["n"] > 0].copy()
print(shown.round(4).to_string(index=False))
print()
print("HGT over-predicts: it places mass at 0.4-0.8 where observed emergence frequency is ~0-0.2.")
print(f"no_graph predicts near the base rate (~{N_TEST_POS / N_TEST:.3f});")
print("most of no_graph's mass falls in [0, 0.1].")
print("This miscalibration drives HGT's worst-in-class share-MAPE and its losing Brier comparison.")
shown

Reliability bins (non-empty), equal-width over [0, 1]:
   model  bin_index  bin_lo  bin_hi   n  mean_predicted  observed_freq
     hgt          1     0.1     0.2   6          0.1761         0.0000
     hgt          2     0.2     0.3  13          0.2493         0.0000
     hgt          3     0.3     0.4  15          0.3627         0.0000
     hgt          4     0.4     0.5  26          0.4545         0.0000
     hgt          5     0.5     0.6  37          0.5558         0.1351
     hgt          6     0.6     0.7  34          0.6450         0.2059
     hgt          7     0.7     0.8   7          0.7334         0.1429
no_graph          0     0.0     0.1 129          0.0263         0.0930
no_graph          1     0.1     0.2   7          0.1383         0.0000
no_graph          3     0.3     0.4   2          0.3570         0.5000

HGT over-predicts: it places mass at 0.4-0.8 where observed emergence frequency is ~0-0.2.
no_graph predicts near the base rate (~0.094);
most of no_graph's mass f

,model,bin_index,bin_lo,bin_hi,n,mean_predicted,observed_freq
1,hgt,1,0.1,0.2,6,0.176052,0.000000
2,hgt,2,0.2,0.3,13,0.249252,0.000000
3,hgt,3,0.3,0.4,15,0.362677,0.000000
4,hgt,4,0.4,0.5,26,0.454515,0.000000
5,hgt,5,0.5,0.6,37,0.555833,0.135135
6,hgt,6,0.6,0.7,34,0.644962,0.205882
7,hgt,7,0.7,0.8,7,0.733360,0.142857
10,no_graph,0,0.0,0.1,129,0.026315,0.093023
11,no_graph,1,0.1,0.2,7,0.138255,0.000000
13,no_graph,3,0.3,0.4,2,0.357012,0.500000


## 6. Figure F3 — `docs/figures/F3_forecasting.png`

In [6]:
# Panel (c) inputs: per-row Brier difference, aligned by (topic_id, origin_year).
h = per_unit[per_unit["model"] == "hgt"].set_index(["topic_id", "origin_year"])
g = per_unit[per_unit["model"] == "no_graph"].set_index(["topic_id", "origin_year"])
common = h.index.intersection(g.index)
brier_h = ((h.loc[common, "emergence_score"] - h.loc[common, "emergent"]) ** 2).to_numpy()
brier_g = ((g.loc[common, "emergence_score"] - g.loc[common, "emergent"]) ** 2).to_numpy()
brier_diff = brier_h - brier_g

order = ["naive", "arima", "mlp", "no_graph", "hgt"]
auc_by = metrics.set_index("model")["emergence_auc"].reindex(order)
ref = auc_ng + 0.05  # the +5pp-over-best-baseline bar HGT had to clear

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

# (a) emergence-AUC bars
colors = ["#9aa0a6", "#9aa0a6", "#9aa0a6", "#4285f4", "#ea4335"]
axes[0].bar(order, auc_by.values, color=colors)
axes[0].axhline(ref, ls="--", c="k", lw=1)
axes[0].axhline(0.5, ls=":", c="#bbbbbb", lw=1)
axes[0].text(
    0.02, ref + 0.006, f"+5pp bar = {ref:.3f}", transform=axes[0].get_yaxis_transform(), fontsize=8
)
axes[0].set_ylim(0.35, 0.9)
axes[0].set_ylabel("emergence AUC (test)")
axes[0].set_title(
    f"(a) Emergence AUC\nHGT {auc_hgt:.3f} < no_graph {auc_ng:.3f} (delta {delta_pp:+.1f}pp)",
    fontsize=9,
)
axes[0].tick_params(axis="x", rotation=30)

# (b) reliability
for m, c, mk in [("hgt", "#ea4335", "o"), ("no_graph", "#4285f4", "s")]:
    d = calib[(calib["model"] == m) & (calib["n"] > 0)]
    axes[1].plot(d["mean_predicted"], d["observed_freq"], marker=mk, c=c, label=m, lw=1.2)
axes[1].plot([0, 1], [0, 1], ls="--", c="k", lw=1, label="perfect")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].set_xlabel("mean predicted")
axes[1].set_ylabel("observed frequency")
axes[1].set_title("(b) Reliability\nHGT below diagonal = over-prediction", fontsize=9)
axes[1].legend(fontsize=8)

# (c) paired Brier difference
axes[2].hist(brier_diff, bins=20, color="#9aa0a6", edgecolor="white")
axes[2].axvline(0, ls=":", c="#888888", lw=1)
axes[2].axvline(
    float(np.median(brier_diff)),
    ls="--",
    c="#ea4335",
    lw=1.5,
    label=f"median {float(np.median(brier_diff)):+.3f}",
)
axes[2].set_xlabel("per-row Brier(HGT) - Brier(no_graph)")
axes[2].set_ylabel("count")
axes[2].set_title(
    f"(c) Paired Brier diff\nWilcoxon p={pb['pvalue']:.1e} (positive = HGT worse)", fontsize=9
)
axes[2].legend(fontsize=8)

fig.suptitle(
    "F3 — 3-year emergence forecasting on the sealed test set (Gate G4: NULL finding)", fontsize=12
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
F3_FIG = FIGURES_DIR / "F3_forecasting.png"
fig.savefig(F3_FIG, dpi=DPI)
plt.close(fig)
print("wrote", F3_FIG)

wrote /Users/samersalman/Desktop/SciField/docs/figures/F3_forecasting.png


In [7]:
# Enforce the < 1 MB figure budget.
sz = F3_FIG.stat().st_size
print(f"F3 size = {sz / 1024:.1f} KB  (dpi={DPI})")
assert sz < 1_000_000, f"F3 too large: {sz} bytes"
print("F3 size assertion PASS — under 1 MB")

F3 size = 87.6 KB  (dpi=120)
F3 size assertion PASS — under 1 MB


In [8]:
# Provenance sidecar for the figure (mirrors notebook 08's F2 pattern): fold in the
# source artifact's config_hash + git_sha so the figure is traceable to the gnn-eval run.
src_hash = metrics_sidecar["config_hash"] if metrics_sidecar else None
src_sha = metrics_sidecar["git_sha"] if metrics_sidecar else None
sidecar_out = record_run(
    artifact_path=F3_FIG,
    inputs={
        "metrics": DATA / "forecasting_test_metrics.parquet",
        "wilcoxon": DATA / "forecasting_test_wilcoxon.json",
    },
    config={
        "figure": "F3_forecasting",
        "session": "V1-S14",
        "dpi": DPI,
        "n_test": N_TEST,
        "n_test_pos": N_TEST_POS,
        "auc_hgt": auc_hgt,
        "auc_no_graph": auc_ng,
        "ablation_delta_pp": delta_pp,
        "brier_pvalue": float(pb["pvalue"]),
        "brier_direction": pb["direction"],
        "overall_pass": bool(vd["overall_pass"]),
        "source_metrics_config_hash": src_hash,
        "source_metrics_git_sha": src_sha,
    },
)
print("recorded run sidecar:", sidecar_out)
print("F3 exists:", F3_FIG.exists(), "| KB:", f"{F3_FIG.stat().st_size / 1024:.1f}")

recorded run sidecar: /Users/samersalman/Desktop/SciField/docs/figures/F3_forecasting.png.run.json
F3 exists: True | KB: 87.6


## 7. Sensitivity (descriptive only — cannot convert fail→pass)

In [9]:
print("HGT emergence AUC under alternate pre-registered labels (descriptive robustness):")
print(sens.round(4).to_string(index=False))
print()
print("The null is robust: HGT AUC is no higher — indeed lower — under every alternate label.")
print("Limitation (logged deviation): the include-noise DENOMINATOR variant was not recomputed in")
print("this gate run; leaf_only is the pre-registered primary denominator. Descriptive only — it")
print("cannot convert a fail into a pass (PR2 §9 / Gate G4).")
sens

HGT emergence AUC under alternate pre-registered labels (descriptive robustness):
      variant         label_column  emergence_auc  n_test  n_test_pos  delta_vs_primary_pp
      primary             emergent         0.7809     138          13               0.0000
additive_jump    emergent_additive         0.6315     138          30             -14.9442
  count_surge emergent_count_surge         0.7068     138           5              -7.4156

The null is robust: HGT AUC is no higher — indeed lower — under every alternate label.
Limitation (logged deviation): the include-noise DENOMINATOR variant was not recomputed in
this gate run; leaf_only is the pre-registered primary denominator. Descriptive only — it
cannot convert a fail into a pass (PR2 §9 / Gate G4).


,variant,label_column,emergence_auc,n_test,n_test_pos,delta_vs_primary_pp
0,primary,emergent,0.780923,138,13,0.000000
1,additive_jump,emergent_additive,0.631481,138,30,-14.944160
2,count_surge,emergent_count_surge,0.706767,138,5,-7.415616
